#### Libraries

In [ ]:
import pandas as pd
import os
import re
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.dates as mdates
import numpy as np
import matplotlib.patches as mpatches


#### Outlet-Level Thermal Analysis

**Goal:** Build an outlet-level, day-granular timeline across CTD, CLG, PLE, and Session logs to study thermal anomalies and PLE/ple spikes.  

#### 1) Dataset schema audit
**Goal:** Inspect columns, dtypes and a few rows.  

In [ ]:
# 📁 Define paths
DATA_DIR = os.path.join("..", "data_all")

In [ ]:
ctd_df = pd.read_csv(os.path.join(DATA_DIR, "CTD_last_year.csv"))
# clg_df = pd.read_csv(os.path.join(DATA_DIR, "CLG_total.xlsx"))
ple_df = pd.read_csv(os.path.join(DATA_DIR, "PLE_last_year.csv"))
sess_df = pd.read_csv(os.path.join(DATA_DIR, "SeccSessionStop_last_year.csv"))


# 🔍 Function to summarize
def summarize_dataset(name, df):
    print(f"\n=== 🔎 {name} Dataset ===")
    print("🧾 Columns:", df.columns.tolist())
    print("📊 Dtypes:")
    print(df.dtypes)
    print("👀 Head:")
    display(df.head(5))

# 📥 Load and summarize each
for name, df in [
    ("CTD", ctd_df),
    # ("CLG", clg_df),
    ("PLE", ple_df),
    ("SeccSessionStop", sess_df)
]:
    summarize_dataset(name, df)


#### 2) PLE: Extract outlet from @message
**Purpose:** Parse outlet id (DC1/DC2/…) from PLE logs.  

In [ ]:
# # 📁 Load dataset
# DATA_DIR = os.path.join("..", "data_all")
# ple_path = os.path.join(DATA_DIR, "PLE.csv")
# ple_df = pd.read_csv(ple_path)

# # ✅ Step 1: Parse outlet info from `@message`
# ple_df["outlet"] = ple_df["@ptr"].str.extract(r"(DC\d+)_CableTempSensor")

# # 🔍 Optional check: Print unique outlets
# print("✅ Extracted outlets:", ple_df["outlet"].dropna().unique())

# # ✅ Step 2: Save the updated file back
# ple_df.to_csv(ple_path, index=False)
# print("✅ PLE_timeseries.xlsx updated with 'outlet' column.")


- 3) Normalize CTD outlets
**Purpose:** Extract integer outlet id from `cableid`.  
- 4) Normalize CLG outlets
**Purpose:** Extract integer outlet id from `cableid`.
- 5) Normalize PLE outlets
**Purpose:** Ensure integer `outlet` in PLE.  
- 6) Normalize Session outlets
**Purpose:** Strip quotes and parse `outlet` to integer.  

In [ ]:
# === Step 1: Split IDOutlet into @logStream and outlet ===

def split_idoutlet(id_str):
    """
    Split IDOutlet into charger (@logStream) and outlet (last digit).
    """
    if pd.isna(id_str):
        return (pd.NA, pd.NA)
    s = str(id_str)
    if s[-1].isdigit():
        return (s[:-1], int(s[-1]))
    else:
        return (s, pd.NA)

# --- Apply to CTD dataset ---
ctd_split = ctd_df["IDOutlet"].apply(split_idoutlet)
ctd_df["@logStream"] = ctd_split.apply(lambda x: x[0])
ctd_df["outlet"] = ctd_split.apply(lambda x: x[1]).astype("Int64")

# --- Apply to PLE dataset ---
ple_split = ple_df["IDOutlet"].apply(split_idoutlet)
ple_df["@logStream"] = ple_split.apply(lambda x: x[0])
ple_df["outlet"] = ple_split.apply(lambda x: x[1]).astype("Int64")


# Verify cableid consistency (last digit-CTD)
if "cableid" in ctd_df.columns:
    ctd_df["cableid_outlet"] = ctd_df["cableid"].apply(lambda x: int(str(x)[-1]) if pd.notna(x) and str(x)[-1].isdigit() else pd.NA).astype("Int64")
    mismatches = ctd_df[ctd_df["outlet"] != ctd_df["cableid_outlet"]]
    if not mismatches.empty:
        print("⚠️ Mismatches found between IDOutlet and cableid:")
        display(mismatches.head(10))
    else:
        print("✅ All CTD rows consistent: IDOutlet and cableid match.")

# --- Apply to SeccSessionStop dataset ---
sess_split = sess_df["IDOutlet"].apply(split_idoutlet)
sess_df["@logStream"] = sess_split.apply(lambda x: x[0])
sess_df["outlet"] = sess_split.apply(lambda x: x[1]).astype("Int64")

print("✅ @logStream + outlet columns created for CTD and SeccSessionStop.")
print("CTD sample:", ctd_df[["@logStream", "outlet"]].head())
print("PLE sample:", ple_df[["@logStream", "outlet"]].head())
print("Sessions sample:", sess_df[["@logStream", "outlet"]].head())


In [ ]:
for df in [ctd_df, sess_df, ple_df]:
    df["@logStream"] = df["@logStream"].astype(str).str.strip()


#### 7) Add day granularity
- All datasets now have a `day` column. Day-level aggregation reduces noise and aligns signals

In [ ]:
#7 Add day column
for df in [ctd_df, sess_df, ple_df]:
    df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors='coerce')
    df["day"] = df["@timestamp"].dt.floor("D")

#### 8) Safety: sort timestamps
**Purpose:** Sort by `@timestamp` before merge. 

In [ ]:
# After loading and timestamp conversion:
for df in [ctd_df, sess_df, ple_df]:
    df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors='coerce')
    df.sort_values(by="@timestamp", inplace=True)
    df["day"] = df["@timestamp"].dt.floor("D")

- 9) CTD aggregation: Purpose:** Aggregate CTD by outlet-day.  
- 10) CLG aggregation
- 11) PLE aggregation
- 12) Session aggregation

In [ ]:
#9
ctd_summary = ctd_df.groupby(["@logStream", "outlet", "day"]).agg(
    CTD_count=("@timestamp", "count"),
    CTD_diff_mean=("diff", "mean"),
    CTD_factor_mean=("factor", "mean"),
    CTD_current_mean=("nowcur", "mean")
).reset_index()

In [ ]:
print(ctd_summary.head())

In [ ]:
# #10
# clg_summary = clg_df.groupby(["@logStream", "outlet", "day"]).agg(
#     CLG_count=("@timestamp", "count"),
#     CLG_diff_mean=("diff", "mean"),
#     CLG_temp1_mean=("Contacttemp", "mean"),
#     CLG_temp2_mean=("Contacttemp2", "mean"),
#     CLG_current_mean=("nowcur", "mean")
# ).reset_index()

In [ ]:
#11
ple_summary = ple_df.groupby(["@logStream", "outlet", "day"]).agg(
    PLE_count=("@timestamp", "count")
).reset_index()

In [ ]:
#12 Re-aggregate Sessions with New features(Duration)
sess_summary = sess_df.groupby(["@logStream", "outlet", "day"]).agg(
    Sess_count=("@timestamp", "count"),
    Sess_energy_mean=("energy", "mean"),
    Sess_temp_diff_mean=("diff", "mean"),
    # NEW:
    Sess_duration_mean=("duration", "mean"),   # avg duration per day (e.g., seconds or minutes)
    Sess_duration_total=("duration", "sum")    # total duration per day
).reset_index()

#### 13) Build master logstream-outlet-timeline
**Purpose:** Merge CTD, CLG, PLE, Sessions on logstream-outlet-day.  

In [ ]:
master_timeline = (
    ctd_summary.merge(sess_summary, on=["@logStream", "outlet", "day"], how="outer")
               .merge(ple_summary, on=["@logStream", "outlet", "day"], how="outer")
)

# Preview
print("✅ Outlet-normalized timeline built!")
print(master_timeline.shape)
display(master_timeline.head())

- clean_master_timeline

- Counts (*_count) → fill missing with 0 (no event that day).

- Means (*_mean) → keep NaN (no measurement; don’t invent zeros).

- Make sure numeric columns are really numeric.

In [ ]:
def clean_master_timeline(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- 1) Identify feature columns by suffix
    count_cols = [c for c in df.columns if c.endswith("_count")]
    mean_cols  = [c for c in df.columns if c.endswith("_mean")]

    # --- 2) Force numeric on feature columns (bad strings -> NaN)
    for c in count_cols + mean_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # --- 3) Fill counts with 0 (no event on that day)
    df[count_cols] = df[count_cols].fillna(0).astype("float64")

    # --- 4) Mask *means* by their family’s count (0 or NaN -> mean must be NaN)
    families = {
        "CTD": ["CTD_diff_mean", "CTD_factor_mean", "CTD_current_mean"],
        # "CLG": ["CLG_diff_mean", "CLG_temp1_mean", "CLG_temp2_mean", "CLG_current_mean"],
        # PLE has only counts
    }

    for fam, cols in families.items():
        cnt = f"{fam}_count"
        if cnt in df.columns:
            mask = (df[cnt].isna()) | (df[cnt] == 0)
            for col in cols:
                if col in df.columns:
                    # Any day with count==0/NaN cannot have a valid mean -> set to NaN
                    df.loc[mask, col] = np.nan

    # --- 5) Ensure key columns typed correctly
    if "@timestamp" in df.columns:
        df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors="coerce")
    if "day" in df.columns:
        df["day"] = pd.to_datetime(df["day"], errors="coerce")
    if "outlet" in df.columns:
        df["outlet"] = pd.to_numeric(df["outlet"], errors="coerce").astype("Int64")

    return df

# 👉 apply right after the outer merges
master_timeline = clean_master_timeline(master_timeline)

print("✅ Cleaned + masked timeline.")
print("Counts (zero-filled):", [c for c in master_timeline.columns if c.endswith("_count")])
print("Means (masked by counts):", [c for c in master_timeline.columns if c.endswith("_mean")])

- convert ms -> minutes BEFORE grouping


In [ ]:
#convert ms -> minutes BEFORE grouping
sess_df["duration"] = pd.to_numeric(sess_df["duration"], errors="coerce")
sess_df["duration_min"] = sess_df["duration"] / 60000.0  # ms to minutes
# then aggregate on "duration_min" instead of "duration"

In [ ]:
print(master_timeline.shape)
display(master_timeline.head(400))

#### 14) Completeness check
**Purpose:** Count NaNs per feature.  

In [ ]:
# 📝 List of columns to check
columns_to_check = [
   "CTD_count","Sess_count", "PLE_count"
]

# 🔍 Count zeros in each
zero_counts = (master_timeline[columns_to_check] == 0).sum()

# 🔢 Also show total number of rows for reference
total_rows = len(master_timeline)

# 🖨️ Print report
print(f"Total rows: {total_rows}\n")
for col in columns_to_check:
    count = int(zero_counts[col])
    pct = round(100 * count / total_rows, 2) if total_rows else 0
    if count == total_rows:
        print(f"⚠️ Column '{col}' has ALL {count} zeros ❗")
    else:
        print(f"✅ Column '{col}' has {count} zeros ({pct}%)")


- Why did Sess_count become 0?
- In the outer merge, if on a given day + charger + outlet there were no sessions recorded, the merged row is created anyway (due to CTD/CLG/PLE presence, or just the calendar alignment).

In [ ]:
# # Investigate rows where Sess_count == 0
# sess_zero = master_timeline[master_timeline["Sess_count"] == 0]

# print("Number of Sess_count == 0 rows:", len(sess_zero))
# print("\nUnique chargers with Sess_count == 0:")
# print(sess_zero["@logStream"].unique())

# print("\nDate range of Sess_count == 0 rows:")
# print(sess_zero["day"].min(), "->", sess_zero["day"].max())

# # Show some sample rows
# display(sess_zero.head(270))

In [ ]:
# print(master_timeline.shape)
# display(master_timeline.head(7000))

#### 15) Alias timeline

##### Filter the Outlets based on those have Issues
-  === Step 1:check Master_file_Outlet_issues for fileting the outlets
-  === Step 2: Clean filter file so it matches main dataset structure


In [ ]:
filter_path = "../data_all/Master_file_Outlet_issues.xlsx"
filter_df = pd.read_excel(filter_path)

# Show schema
print("=== Filter Excel File ===")
print("🧾 Columns:", filter_df.columns.tolist())
print("\n📊 Dtypes:")
print(filter_df.dtypes)

print("\n Sample rows:")
display(filter_df.head())
print(filter_df.count())

#Rename columns to align with CTD / Sessions /  PLE
filter_df = filter_df.rename(columns={
    "ID": "@logStream",
    "Outlet": "outlet"
})

# Ensure outlet is integer type
filter_df["outlet"] = filter_df["outlet"].astype("Int64")

print("✅ Filter file cleaned.")
print(filter_df.head())


##### 🔄 Filtering + Aggregation + Master Timeline (correct order)


In [ ]:
##### 🔄 Filtering + Aggregation + Master Timeline (correct order)

# 1) Normalize charger IDs everywhere first
for df in [ctd_df, sess_df, ple_df]:
    df["@logStream"] = df["@logStream"].astype(str).str.strip().str.lower()
filter_df["@logStream"] = filter_df["@logStream"].astype(str).str.strip().str.lower()

# 2) Filter datasets by valid charger–outlet pairs
valid_pairs = set(zip(filter_df["@logStream"], filter_df["outlet"]))
print(f"✅ Loaded {len(valid_pairs)} valid charger–outlet pairs from filter file.")

ctd_filtered = ctd_df[ctd_df[["@logStream", "outlet"]].apply(tuple, axis=1).isin(valid_pairs)].copy()
sess_filtered = sess_df[sess_df[["@logStream", "outlet"]].apply(tuple, axis=1).isin(valid_pairs)].copy()
ple_filtered  = ple_df[ple_df[["@logStream", "outlet"]].apply(tuple, axis=1).isin(valid_pairs)].copy()

# just a trick to avoide filtering based on the Niklas model output!, to get back, remove the comment above and remove the these three lines below
# ctd_filtered = ctd_df
# sess_filtered = sess_df
# ple_filtered = ple_df

print("✅ Filtering complete.")
print("CTD rows before:", len(ctd_df), "→ after:", len(ctd_filtered))
print("Sessions rows before:", len(sess_df), "→ after:", len(sess_filtered))
print("PLE rows before:", len(ple_df), "→ after:", len(ple_filtered))

# 3) Aggregations
ctd_summary = ctd_filtered.groupby(["@logStream", "outlet", "day"]).agg(
    CTD_count=("@timestamp", "count"),
    CTD_diff_mean=("diff", "mean"),
    CTD_factor_mean=("factor", "mean"),
    CTD_current_mean=("nowcur", "mean")
).reset_index()

sess_summary = sess_filtered.groupby(["@logStream", "outlet", "day"]).agg(
    Sess_count=("@timestamp", "count"),
    Sess_energy_mean=("energy", "mean"),
    Sess_temp_diff_mean=("diff", "mean"),
    Sess_duration_mean=("duration_min", "mean"),
    Sess_duration_total=("duration_min", "sum")
).reset_index()

ple_summary = ple_filtered.groupby(["@logStream", "outlet", "day"]).agg(
    PLE_count=("@timestamp", "count")
).reset_index()

# 4) Build master timeline
master_timeline = (
    ctd_summary.merge(sess_summary, on=["@logStream", "outlet", "day"], how="outer")
               .merge(ple_summary, on=["@logStream", "outlet", "day"], how="outer")
)
master_timeline = clean_master_timeline(master_timeline)

# 5) Alias for downstream use
outlet_timeline = master_timeline

print("✅ Master timeline built + cleaned")
print("Unique outlets after filtering:", outlet_timeline.groupby(["@logStream", "outlet"]).ngroups)

# 6) Derived Features (add AFTER cleaning)
outlet_timeline["Energy_per_session"] = np.where(
    outlet_timeline["Sess_count"] > 0,
    outlet_timeline["Sess_energy_mean"] / outlet_timeline["Sess_count"],
    np.nan
)

outlet_timeline["CTD_per_session"] = np.where(
    outlet_timeline["Sess_count"] > 0,
    outlet_timeline["CTD_count"] / outlet_timeline["Sess_count"],
    np.nan
)

outlet_timeline["Energy_per_duration"] = np.where(
    outlet_timeline["Sess_duration_mean"] > 0,
    outlet_timeline["Sess_energy_mean"] / outlet_timeline["Sess_duration_mean"],
    np.nan
)

outlet_timeline["PLE_per_session"] = np.where(
    outlet_timeline["Sess_count"] > 0,
    outlet_timeline["PLE_count"] / outlet_timeline["Sess_count"],
    np.nan
)

print("✅ Derived features added:", 
      ["Energy_per_session", "CTD_per_session", "Energy_per_duration", "PLE_per_session"])


#### New features are added here:
- CTD per session
- Energy per duration
- PLE per session

In [ ]:
print("CTD unique pairs:", ctd_filtered.groupby(["@logStream", "outlet"]).ngroups)
print("Sess unique pairs:", sess_filtered.groupby(["@logStream", "outlet"]).ngroups)
print("PLE unique pairs:", ple_filtered.groupby(["@logStream", "outlet"]).ngroups)

In [ ]:
print("Unique @logStream count CTD:", ctd_df["@logStream"].nunique())
print("Unique @logStream count Sess:", sess_df["@logStream"].nunique())
print("Unique @logStream count PLE:", ple_df["@logStream"].nunique())

In [ ]:
print("Unique outlets after filtering:", 
      outlet_timeline.groupby(["@logStream", "outlet"]).ngroups)

### Visualization function
#### 16) Batch export plots - All outlets
- Auto-save PNG per charger-outlet.  

In [ ]:
# === Trend helper (with rise/fall markers) ===
def _add_trends(ax, x_dates, y, rolling_window=21, base_color="C0", label="Series"):
    """
    Plot raw series and its rolling mean trend on the given axis.
    Adds markers when a sustained upward or downward trend starts.
    """
    ax.plot(x_dates, y, marker='o', linestyle='-', color=base_color, alpha=0.5, label=label)

    y_roll = pd.Series(y, index=pd.to_datetime(x_dates)).rolling(
        window=rolling_window, min_periods=1
    ).mean()

    ax.plot(x_dates, y_roll.values, linestyle='--', linewidth=2,
            color="black", label=f"Trend ({rolling_window}d rolling)")

    # Skip rise/fall markers if too few points
    if len(y_roll.dropna()) < 5:
        return

    sign = np.sign(y_roll.diff().fillna(0).values)
    for i in range(len(sign) - 3):
        if all(sign[i:i+3] > 0):  # upward
            ax.annotate("↑ rise", (x_dates.iloc[i], y_roll.iloc[i]),
                        xytext=(0, 10), textcoords="offset points",
                        color=base_color, fontsize=9,
                        arrowprops=dict(arrowstyle="->", color=base_color))
            break


# === Outlier capping helper ===
def cap_outliers(series, upper_quantile=0.99):
    s = pd.to_numeric(series, errors="coerce")
    if s.dropna().empty:
        return s
    cap_value = s.quantile(upper_quantile)
    return np.minimum(s, cap_value)


# === Updated export function with score & rank in title/filename ===
def export_all_outlets_with_trends(outlet_timeline,
                                   ranking_df=None,
                                   output_dir="../plots_alloutlet_timelines_Grouped",
                                   rolling_window=30,
                                   duration_unit_label="min"):
    import os
    os.makedirs(output_dir, exist_ok=True)

    for (charger, outlet), df in outlet_timeline.groupby(["@logStream", "outlet"]):
        if df.empty:
            continue

        df = df.sort_values("day").copy()
        df["day"] = pd.to_datetime(df["day"], errors="coerce")

        # --- Apply outlier capping ---
        features_to_cap = {
            "Energy_per_session": 0.90,
            "Energy_per_duration": 0.90,
            "Sess_temp_diff_mean": 0.90,
            "Sess_duration_mean": 0.90,
            "CTD_count": 0.99,
            "PLE_count": 0.99,
            "CTD_per_session": 0.90,
            "PLE_per_session": 0.90,
        }
        for col, q in features_to_cap.items():
            if col in df:
                df[col] = cap_outliers(df[col], q)

        # --- Lookup score & rank ---
        score_str, rank_str = "", ""
        filename_suffix = "_grouped"
        if ranking_df is not None:
            row = ranking_df[(ranking_df["@logStream"] == charger) &
                             (ranking_df["outlet"] == outlet)]
            if not row.empty:
                score = row["Score"].values[0]
                rank = row["Rank"].values[0]
                score_str = f" | Score: {score:.1f}"
                rank_str  = f" | Rank: {rank}"
                filename_suffix = f"_Rank{rank}"

        # --- 5 panels ---
        fig, axs = plt.subplots(5, 1, figsize=(18, 20), sharex=True)
        fig.suptitle(f"Timeline with Trends: {charger} – Outlet {outlet}{score_str}{rank_str}",
                     fontsize=16)

        # 1) Energy Usage (Wh/session + Wh/min)
        if "Energy_per_session" in df:
            _add_trends(axs[0], df["day"], df["Energy_per_session"].to_numpy(dtype=float),
                        rolling_window, base_color="green", label="Wh/Session")
        if "Energy_per_duration" in df:
            _add_trends(axs[0], df["day"], df["Energy_per_duration"].to_numpy(dtype=float),
                        rolling_window, base_color="darkgreen", label=f"Wh/{duration_unit_label}")
        axs[0].set_title("Energy Usage")
        axs[0].set_ylabel("Wh"); axs[0].legend(loc="upper left")

        # 2) Session Temperature Diff
        if "Sess_temp_diff_mean" in df:
            _add_trends(axs[1], df["day"], df["Sess_temp_diff_mean"].to_numpy(dtype=float),
                        rolling_window, base_color="orange", label="Temp Diff (°C)")
        axs[1].set_title("Session Temperature Diff (°C)")
        axs[1].set_ylabel("°C"); axs[1].legend(loc="upper left")

        # 3) Counts (CTD + PLE)
        if "CTD_count" in df:
            _add_trends(axs[2], df["day"], df["CTD_count"].to_numpy(dtype=float),
                        rolling_window, base_color="brown", label="CTD Count")
        if "PLE_count" in df:
            _add_trends(axs[2], df["day"], df["PLE_count"].to_numpy(dtype=float),
                        rolling_window, base_color="blue", label="PLE Count")
        axs[2].set_title("Event Counts")
        axs[2].set_ylabel("count"); axs[2].legend(loc="upper left")

        # 4) Ratios (CTD/session + PLE/session)
        if "CTD_per_session" in df:
            _add_trends(axs[3], df["day"], df["CTD_per_session"].to_numpy(dtype=float),
                        rolling_window, base_color="red", label="CTD per Session")
        if "PLE_per_session" in df:
            _add_trends(axs[3], df["day"], df["PLE_per_session"].to_numpy(dtype=float),
                        rolling_window, base_color="blue", label="PLE per Session")
        axs[3].axhline(1, color="gray", linestyle="--", alpha=0.6)
        axs[3].set_title("Ratios (per Session)")
        axs[3].set_ylabel("ratio"); axs[3].legend(loc="upper left")

        # 5) Session Duration
        if "Sess_duration_mean" in df:
            _add_trends(axs[4], df["day"], df["Sess_duration_mean"].to_numpy(dtype=float),
                        rolling_window, base_color="purple", label=f"Duration ({duration_unit_label})")
        axs[4].set_title(f"Session Duration (mean, {duration_unit_label})")
        axs[4].set_ylabel(duration_unit_label); axs[4].legend(loc="upper left")

        # Format x-axis
        axs[4].xaxis.set_major_locator(mdates.MonthLocator())
        axs[4].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        plt.xticks(rotation=45)

        plt.tight_layout(rect=[0, 0.02, 1, 0.97])

        # Save plot with rank suffix if available
        filename = f"{charger}_Outlet{outlet}{filename_suffix}.png".replace("/", "_")
        plt.savefig(os.path.join(output_dir, filename), dpi=150)
        plt.close(fig)

    print(f"✅ Batch export done: grouped plots saved to {output_dir}")


In [ ]:
ranking_df = score_outlets_combined(outlet_timeline, rolling_window=60)

if "Final_score" in ranking_df.columns:
    ranking_df = ranking_df.rename(columns={"Final_score": "Score"})

ranking_df = ranking_df.sort_values("Score", ascending=False).reset_index(drop=True)
ranking_df["Rank"] = ranking_df.index + 1


#### 17) Rule-based scoring

In [ ]:
#V2
# === Updated Scoring Function v2.2 (safe gradient) ===
def score_outlets_combined(outlet_timeline, rolling_window=60, min_sessions=250):
    """
    Score outlets based on temperature, CTD, PLE, and stability rules.
    v2.2: safe gradient (handles very short series).
    """

    results = []

    for (charger, outlet), df in outlet_timeline.groupby(["@logStream", "outlet"]):
        if df.empty:
            continue

        df = df.sort_values("day").copy()
        df["day"] = pd.to_datetime(df["day"], errors="coerce")

        # Session evidence
        sess_total = df["Sess_count"].sum()

        # Helper: safe slope
        def safe_slope(series):
            series = series.fillna(0)
            if len(series) < 2:
                return np.zeros(len(series))
            return np.gradient(series)

        # --- A) Temperature Score ---
        temp = df["Sess_temp_diff_mean"]
        temp_med = temp.tail(90).median(skipna=True)
        temp_slope = safe_slope(temp)
        temp_sustained_rise = (temp_slope[-90:] > 0).mean() > 0.6 if len(temp) >= 90 else False

        temp_score = 0
        if temp_med > 10: temp_score += 3
        elif temp_med > 7: temp_score += 2
        elif temp_med > 5: temp_score += 1
        if temp_sustained_rise: temp_score += 3

        # Burst rules
        for window, thresh in [(180, 0.3), (30, 0.8), (7, 2.0)]:
            if len(temp) >= window:
                start, end = temp.iloc[-window], temp.iloc[-1]
                if pd.notna(start) and pd.notna(end) and start > 0:
                    if (end - start) / start > thresh:
                        temp_score += 2
                        break

        # --- B) CTD Score ---
        ctd = df["CTD_count"]
        ctd_med = ctd.tail(90).median(skipna=True)
        ctd_slope = safe_slope(ctd)
        ctd_sustained = (ctd_slope[-60:] > 0).mean() > 0.6 if len(ctd) >= 60 else False

        ctd_score = 0
        if ctd_sustained: ctd_score += 2
        for window, thresh in [(180, 0.3), (30, 0.8), (7, 2.0)]:
            if len(ctd) >= window:
                start, end = ctd.iloc[-window], ctd.iloc[-1]
                if pd.notna(start) and pd.notna(end) and start > 0:
                    if (end - start) / start > thresh:
                        ctd_score += 1
                        break
        if ctd_med > 10: ctd_score += 1

        # --- C) PLE Score ---
        ple = df["PLE_count"]
        ple_med = ple.tail(90).median(skipna=True)
        ple_slope = safe_slope(ple)
        ple_sustained = (ple_slope[-60:] > 0).mean() > 0.6 if len(ple) >= 60 else False

        ple_score = 0
        if ple_sustained: ple_score += 2
        for window, thresh in [(180, 0.3), (30, 0.8), (7, 2.0)]:
            if len(ple) >= window:
                start, end = ple.iloc[-window], ple.iloc[-1]
                if pd.notna(start) and pd.notna(end) and start > 0:
                    if (end - start) / start > thresh:
                        ple_score += 1
                        break
        if ple_med > 1: ple_score += 1

        # --- D) Stability Score ---
        dur = df["Sess_duration_mean"]
        dur_slope = safe_slope(dur)
        stab_score = 0
        if temp_sustained_rise and (dur_slope[-60:] < 0).mean() > 0.6:
            stab_score += 2

        # --- Total Score ---
        total_score = temp_score + ctd_score + ple_score + stab_score

        # Session evidence weight
        if sess_total >= 1000: weight = 1.0
        elif sess_total >= 500: weight = 0.8
        elif sess_total >= 250: weight = 0.6
        else: weight = 0.4

        final_score = total_score * weight

        results.append({
            "@logStream": charger,
            "outlet": outlet,
            "Temp_score": temp_score,
            "CTD_score": ctd_score,
            "PLE_score": ple_score,
            "Stability_score": stab_score,
            "Sess_count_total": sess_total,
            "Final_score": final_score
        })

    return pd.DataFrame(results).sort_values("Final_score", ascending=False).reset_index(drop=True)


### Previous scoring

In [ ]:
# # === Updated Scoring Function v2.1 ===
# def score_outlets_combined(outlet_timeline, rolling_window=60, min_sessions=250):
#     """
#     Score outlets based on temperature, CTD, PLE, and stability rules.
#     v2.1: Temperature-driven, CTD/PLE trend-focused.
#     """

#     results = []

#     for (charger, outlet), df in outlet_timeline.groupby(["@logStream", "outlet"]):
#         if df.empty:
#             continue

#         df = df.sort_values("day").copy()
#         df["day"] = pd.to_datetime(df["day"], errors="coerce")

#         # Smooth with rolling mean
#         roll = df.rolling(rolling_window, min_periods=1)

#         # Session evidence
#         sess_total = df["Sess_count"].sum()

#         # --- A) Temperature Score ---
#         temp = df["Sess_temp_diff_mean"]
#         temp_med = temp.tail(90).median(skipna=True)
#         temp_slope = np.gradient(temp.fillna(0))
#         temp_sustained_rise = (temp_slope[-90:] > 0).mean() > 0.6 if len(temp) >= 90 else False

#         temp_score = 0
#         if temp_med > 10: temp_score += 3
#         elif temp_med > 7: temp_score += 2
#         elif temp_med > 5: temp_score += 1
#         if temp_sustained_rise: temp_score += 3

#         # Burst rules
#         for window, thresh in [(180, 0.3), (30, 0.8), (7, 2.0)]:
#             if len(temp) >= window:
#                 start, end = temp.iloc[-window], temp.iloc[-1]
#                 if pd.notna(start) and pd.notna(end) and start > 0:
#                     if (end - start) / start > thresh:
#                         temp_score += 2
#                         break

#         # --- B) CTD Score ---
#         ctd = df["CTD_count"]
#         ctd_med = ctd.tail(90).median(skipna=True)
#         ctd_slope = np.gradient(ctd.fillna(0))
#         ctd_sustained = (ctd_slope[-60:] > 0).mean() > 0.6 if len(ctd) >= 60 else False

#         ctd_score = 0
#         if ctd_sustained: ctd_score += 2
#         for window, thresh in [(180, 0.3), (30, 0.8), (7, 2.0)]:
#             if len(ctd) >= window:
#                 start, end = ctd.iloc[-window], ctd.iloc[-1]
#                 if pd.notna(start) and pd.notna(end) and start > 0:
#                     if (end - start) / start > thresh:
#                         ctd_score += 1
#                         break
#         if ctd_med > 10: ctd_score += 1

#         # --- C) PLE Score ---
#         ple = df["PLE_count"]
#         ple_med = ple.tail(90).median(skipna=True)
#         ple_slope = np.gradient(ple.fillna(0))
#         ple_sustained = (ple_slope[-60:] > 0).mean() > 0.6 if len(ple) >= 60 else False

#         ple_score = 0
#         if ple_sustained: ple_score += 2
#         for window, thresh in [(180, 0.3), (30, 0.8), (7, 2.0)]:
#             if len(ple) >= window:
#                 start, end = ple.iloc[-window], ple.iloc[-1]
#                 if pd.notna(start) and pd.notna(end) and start > 0:
#                     if (end - start) / start > thresh:
#                         ple_score += 1
#                         break
#         if ple_med > 1: ple_score += 1

#         # --- D) Stability Score ---
#         dur = df["Sess_duration_mean"]
#         dur_slope = np.gradient(dur.fillna(0))
#         stab_score = 0
#         if temp_sustained_rise and (dur_slope[-60:] < 0).mean() > 0.6:
#             stab_score += 2

#         # --- Total Score ---
#         total_score = temp_score + ctd_score + ple_score + stab_score

#         # Session evidence weight
#         if sess_total >= 1000: weight = 1.0
#         elif sess_total >= 500: weight = 0.8
#         elif sess_total >= 250: weight = 0.6
#         else: weight = 0.4

#         final_score = total_score * weight

#         results.append({
#             "@logStream": charger,
#             "outlet": outlet,
#             "Temp_score": temp_score,
#             "CTD_score": ctd_score,
#             "PLE_score": ple_score,
#             "Stability_score": stab_score,
#             "Sess_count_total": sess_total,
#             "Final_score": final_score
#         })

#     return pd.DataFrame(results).sort_values("Final_score", ascending=False).reset_index(drop=True)


### Visualization

- Plot only Top-20 outlets

In [ ]:
# === Plot only top-N outlets based on ranking ===
def export_top_outlets(outlet_timeline, ranking_df, n=20, output_dir="../plots_topN"):
    import os
    os.makedirs(output_dir, exist_ok=True)

    # Take the top N from the ranking list
    top_list = ranking_df.sort_values("Score", ascending=False).head(n)

    # Create set of charger–outlet pairs
    selected_pairs = set(zip(top_list["@logStream"], top_list["outlet"]))

    # Filter outlet_timeline to just those pairs
    filtered = outlet_timeline[outlet_timeline[["@logStream", "outlet"]]
                               .apply(tuple, axis=1)
                               .isin(selected_pairs)]

    # Reuse the updated export function and pass ranking_df
    export_all_outlets_with_trends(filtered,
                                   ranking_df=ranking_df,
                                   output_dir=output_dir,
                                   rolling_window=30,
                                   duration_unit_label="min")

    print(f"✅ Exported plots for top {n} outlets into {output_dir}")


In [ ]:
# Build ranking
ranking_df = score_outlets_combined(outlet_timeline, rolling_window=60)
ranking_df = ranking_df.rename(columns={"Final_score": "Score"}) if "Final_score" in ranking_df.columns else ranking_df
ranking_df = ranking_df.sort_values("Score", ascending=False).reset_index(drop=True)
ranking_df["Rank"] = ranking_df.index + 1

# Export top-20 plots with score + rank in title and filename
export_top_outlets(outlet_timeline, ranking_df, n=20, output_dir="../plots_top20_ModelFiltered")


In [ ]:
# === Export top-N chargers as a single string ===
def export_top_chargers(ranking_df, n=20, output_path=None):
    # Take top N chargers
    top_chargers = ranking_df.sort_values("Score", ascending=False).head(n)["@logStream"].unique()

    # Join into pipe-separated string
    charger_str = "|".join(top_chargers)

    print("Top chargers:", charger_str)

    # Optionally save to file
    if output_path:
        with open(output_path, "w") as f:
            f.write(charger_str)

    return charger_str


# Example usage:
charger_str = export_top_chargers(ranking_df, n=20, output_path="../plots_top20_ModelFiltered/top_chargers.txt")


##### Working on new Interpretable visualizations!

In [ ]:
def plot_outlet_interpretable(df, charger, outlet, rolling_window=30, duration_unit_label="min"):
    """
    Interpretable plot with annotations, shaded anomalies, and risk score in title.
    """

    # Filter data
    df = df[(df["@logStream"] == charger) & (df["outlet"] == outlet)].copy()
    if df.empty:
        print(f"No data for {charger} Outlet {outlet}")
        return

    df = df.sort_values("day")
    df["day"] = pd.to_datetime(df["day"], errors="coerce")

    # --- Compute rolling features for smoother patterns ---
    temp_roll = df["Sess_temp_diff_mean"].rolling(rolling_window, min_periods=1).mean()
    dur_roll  = df["Sess_duration_mean"].rolling(rolling_window, min_periods=1).mean()
    ctd_roll  = df["CTD_count"].rolling(rolling_window, min_periods=1).mean()
    ple_roll  = df["PLE_count"].rolling(rolling_window, min_periods=1).mean()

    # --- Scoring (simple version, same rules as before) ---
    score = 0
    temp_score = 0
    ctd_score = 0
    ple_score = 0

    if temp_roll.median(skipna=True) > 5: temp_score += 1
    if (temp_roll > 10).sum() > 10: temp_score += 2
    score += temp_score

    if df["CTD_count"].max() > 50: ctd_score += 2
    if (df["CTD_count"] > 10).sum() > 30: ctd_score += 1
    score += ctd_score

    if (df["PLE_count"] > 5).sum() > 10: ple_score += 1
    if (df["PLE_count"] > 20).sum() > 5: ple_score += 2
    score += ple_score

    # --- Create figure ---
    fig, axs = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
    fig.suptitle(f"{charger} – Outlet {outlet} | Risk Score: {score} "
                 f"(Temp={temp_score}, CTD={ctd_score}, PLE={ple_score})", fontsize=14, fontweight="bold")

    # 1) Temp diff with shaded zones
    axs[0].plot(df["day"], temp_roll, color="orange", label="Temp Diff (°C)")
    axs[0].fill_between(df["day"], 10, axs[0].get_ylim()[1], color="red", alpha=0.2, label=">10°C zone")
    axs[0].set_ylabel("Temp (°C)")
    axs[0].legend(loc="upper left")
    axs[0].set_title("Temperature Difference Trend")

    # 2) Duration
    axs[1].plot(df["day"], dur_roll, color="purple", label=f"Duration ({duration_unit_label})")
    axs[1].set_ylabel(duration_unit_label)
    axs[1].legend(loc="upper left")
    axs[1].set_title("Session Duration Trend")

    # 3) CTD + PLE counts overlay
    axs[2].plot(df["day"], ctd_roll, color="brown", label="CTD Count")
    axs[2].plot(df["day"], ple_roll, color="blue", label="PLE Count")
    # Mark spikes
    axs[2].scatter(df["day"][df["CTD_count"] > 50], df["CTD_count"][df["CTD_count"] > 50],
                   color="red", marker="x", s=50, label="CTD spike > 50")
    axs[2].scatter(df["day"][df["PLE_count"] > 20], df["PLE_count"][df["PLE_count"] > 20],
                   color="black", marker="o", s=40, label="PLE spike > 20")
    axs[2].set_ylabel("Counts")
    axs[2].legend(loc="upper left")
    axs[2].set_title("CTD & PLE Counts")

    # Format x-axis
    axs[2].xaxis.set_major_locator(mdates.MonthLocator())
    axs[2].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.xticks(rotation=45)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    return fig


In [ ]:
fig = plot_outlet_interpretable(outlet_timeline, charger, outlet, rolling_window)
filename = f"{charger}_Outlet{outlet}_interpretable.png".replace("/", "_")
filepath = os.path.join(output_dir, filename)
fig.savefig(filepath, dpi=150)
plt.close(fig)
